In [10]:
import torch.nn as nn
import torch
import torch.optim as optim
import copy
from helper_functions.data_loading.data_utils import get_dataloaders
from torchvision import models
import json
DATA_DIR = "dataset/garbage_classification"

In [2]:

# device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

# data
train_loader, val_loader, test_loader, class_names, class_weights = get_dataloaders(DATA_DIR)

# model
# reuse the same data + device from before; re-run get_dataloaders if needed
# train_loader, val_loader, test_loader, class_names, class_weights = get_dataloaders(DATA_DIR)

weights = models.MobileNet_V2_Weights.IMAGENET1K_V1     # pretrained on ImageNet
tmodel = models.mobilenet_v2(weights=weights)           # first run downloads ~14MB

# replace the final classifier layer: 1280 features -> your 12 classes
tmodel.classifier[1] = nn.Linear(tmodel.last_channel, len(class_names))

# freeze the pretrained body, keep only the new head trainable
for p in tmodel.features.parameters():
    p.requires_grad = False

tmodel = tmodel.to(device)

device: mps
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /Users/mukundanramesh/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:01<00:00, 11.9MB/s]


In [3]:
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
# optimizer sees ONLY the unfrozen params (the new head)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, tmodel.parameters()), lr=1e-3)

In [5]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=15):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        # ---- train ----
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            run_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        train_loss, train_acc = run_loss / total, correct / total

        # ---- validate ----
        model.eval()
        run_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                run_loss += loss.item() * imgs.size(0)
                correct += (outputs.argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_loss, val_acc = run_loss / total, correct / total

        for k, v in zip(history, [train_loss, val_loss, train_acc, val_acc]):
            history[k].append(v)

        print(f"Epoch {epoch:2d}/{epochs} | train_loss {train_loss:.3f} acc {train_acc:.3f} "
              f"| val_loss {val_loss:.3f} acc {val_acc:.3f}")

        if val_acc > best_val_acc:                       # keep the best version, not the last
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_weights)
    print(f"Best val acc: {best_val_acc:.3f}")
    return model, history

In [6]:
tmodel, hist1 = train_model(tmodel, train_loader, val_loader, criterion, optimizer, device, epochs=10)

Epoch  1/10 | train_loss 0.806 acc 0.820 | val_loss 0.443 acc 0.911
Epoch  2/10 | train_loss 0.458 acc 0.885 | val_loss 0.359 acc 0.917
Epoch  3/10 | train_loss 0.399 acc 0.898 | val_loss 0.343 acc 0.915
Epoch  4/10 | train_loss 0.373 acc 0.901 | val_loss 0.357 acc 0.915
Epoch  5/10 | train_loss 0.370 acc 0.902 | val_loss 0.341 acc 0.920
Epoch  6/10 | train_loss 0.357 acc 0.907 | val_loss 0.347 acc 0.920
Epoch  7/10 | train_loss 0.352 acc 0.909 | val_loss 0.324 acc 0.924
Epoch  8/10 | train_loss 0.337 acc 0.910 | val_loss 0.366 acc 0.915
Epoch  9/10 | train_loss 0.326 acc 0.911 | val_loss 0.344 acc 0.920
Epoch 10/10 | train_loss 0.326 acc 0.910 | val_loss 0.303 acc 0.930
Best val acc: 0.930


In [7]:
import os
os.makedirs("models", exist_ok=True)
torch.save(tmodel.state_dict(), "models/mobilenet_phase1.pt")

In [8]:
for p in tmodel.parameters():
    p.requires_grad = True                                   # unfreeze the whole network
optimizer = torch.optim.Adam(tmodel.parameters(), lr=1e-4)   # 10x lower LR = gentle nudges

In [9]:
tmodel, hist2 = train_model(tmodel, train_loader, val_loader, criterion, optimizer, device, epochs=5)

Epoch  1/5 | train_loss 0.328 acc 0.913 | val_loss 0.252 acc 0.942
Epoch  2/5 | train_loss 0.182 acc 0.947 | val_loss 0.223 acc 0.949
Epoch  3/5 | train_loss 0.136 acc 0.961 | val_loss 0.197 acc 0.953
Epoch  4/5 | train_loss 0.097 acc 0.971 | val_loss 0.201 acc 0.957
Epoch  5/5 | train_loss 0.089 acc 0.972 | val_loss 0.177 acc 0.957
Best val acc: 0.957


In [11]:
torch.save(tmodel.state_dict(), "models/mobilenet_v2.pt")
with open("reports/metrics/mobilenet_history.json", "w") as f:
    json.dump({"phase1": hist1, "phase2": hist2}, f, indent=2)